<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/02-ctc-stt-from-scratch/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np

import torch
import torch.nn as nn

import torchaudio
import pandas as pd

from torch.utils.data import Dataset, DataLoader

In [ ]:
# setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# setting seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# config
CFG = {
    "sample_rate": 16000,
    "batch_size": 16,
    "epochs": 100,
    "lr": 5e-4,
    "embedding_dim": 32,
    "num_transformer_layers": 6,
    "num_heads": 4,
    "strides": (2,2,2,2)
}

In [ ]:
# download dataset
from TTS.utils.downloaders import download_ljspeech
dataset_path = download_ljspeech("./data")
print(dataset_path)

In [ ]:
# read metadata
import os
import pandas as pd

path = "./data/LJSpeech-1.1"

metadata = pd.read_csv(
    os.path.join(path, "metadata.csv"),
    sep="|",
    header=None,
    names=["id", "text", "normalized_text"]
)

metadata.head()

In [ ]:
# load audio and text properly
wavs_path = os.path.join(path, "wavs")

audio_paths = [
    os.path.join(wavs_path, f"{fid}.wav") for fid in metadata["id"]
]

texts = [
    t.upper() for t in metadata["normalized_text"]
]

print(audio_paths[0], texts[0])

In [ ]:
import string

# vocab
characters = list(string.ascii_uppercase) + [" "]
blank_token = "<blank>"

vocab = characters + [blank_token]

# mappings
char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

print("Vocab size:", len(vocab))

In [ ]:
# tokenizer
def text_to_tokens(text):
  return [char_to_idx[c] for c in text if c in char_to_idx]

token_sequences = [text_to_tokens(t) for t in texts]

In [ ]:
# build dataset
class STTDataset(Dataset):
  def __init__(self, audio_paths, token_sequences):
    self.audio_paths = audio_paths
    self.token_sequences = token_sequences

  def __len__(self):
    return len(self.audio_paths)

  def __getitem__(self, idx):
    # load audio
    waveform, sr = torchaudio.load(self.audio_paths[idx])

    # mono
    if waveform.shape[0] > 1:
      waveform = waveform.mean(dim=0) # (T)

    # resample to 16k
    if sr != 16000:
      waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    mel = mel_transform(waveform)

    # tokens
    tokens = torch.tensor(self.token_sequences[idx], dtype=torch.long)

    return mel.squeeze(0), tokens

In [ ]:
# collate function (needed for ctc_loss)
# pad time along dim=0 and move it back
def collate_fn(batch):
  mels, tokens = zip(*batch)

  # lengths (before padding)
  input_lengths = torch.tensor([m.shape[1] for m in mels], dtype=torch.long)
  target_lengths = torch.tensor([len(t) for t in tokens], dtype=torch.long)

  # (80, T) -> (T, 80) for padding
  mels = [m.T for m in mels]

  # pad audio
  mels = torch.nn.utils.rnn.pad_sequence(mels, batch_first=True) # (B, T, 80)
  mels = mels.transpose(1, 2) # (B, 80, T)

  # pad tokens
  tokens = torch.nn.utils.rnn.pad_sequence(tokens, batch_first=True) # (B, U)

  return mels, tokens, input_lengths, target_lengths

In [ ]:
# dataloader
dataset = STTDataset(
    audio_paths,
    token_sequences
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# test one batch
# waveforms, tokens, input_lengths, target_lengths = next(iter(loader))

# print("Waveforms:", waveforms.shape)
# print("Tokens:", tokens.shape)
# print("Input lengths:", input_lengths)
# print("Target lengths:", target_lengths)

mels            → (B, 80, T)

tokens          → (B, U)

input_lengths   → (B,)

target_lengths  → (B,)

In [ ]:
# build model
class STTModel(nn.Module):
  def __init__(self, n_mels=80, hidden=256, vocab_size=30):
    super().__init__()

    self.encoder = nn.LSTM(
        input_size = n_mels,
        hidden_size = hidden,
        num_layers = 2,
        batch_first = True,
        bidirectional = True
    )

    self.fc = nn.Linear(hidden*2, vocab_size)

  def forward(self, x):
    # x: (B, 80, T)
    x = x.transpose(1, 2) # (B, T, 80)
    out, _  = self.encoder(x) # (B, T, 2H)
    out = self.fc(out) # (B, T, vocab_size)
    out = out.log_softmax(dim=-1)
    return out

In [ ]:
# initialize model
vocab_size = len(vocab)

model = STTModel(
    n_mels=80,
    hidden=256,
    vocab_size=vocab_size
).to(device)

In [ ]:
# training loop

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
blank_idx = char_to_idx["<blank>"]
ctc_loss = torch.nn.CTCLoss(blank=blank_idx)

num_epochs = 60
best_loss = float("inf")
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(num_epochs):

  model.train()
  total_loss = 0.0

  for i, (mels, tokens, input_lengths, target_lengths) in enumerate(loader):

        mels = mels.to(device)                     # (B, 80, T)
        tokens = tokens.to(device)
        input_lengths = input_lengths.to(device)
        target_lengths = target_lengths.to(device)

        # forward
        outputs = model(mels)

        # CTC expects (T, B, vocab_size)
        outputs = outputs.permute(1, 0, 2)

        # loss
        loss = ctc_loss(outputs.log_softmax(2), tokens, input_lengths, target_lengths)

        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient clipping (for LSTM stability)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total_loss += loss.item()

        if i % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Step [{i}] Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)

    print(f"\nEpoch [{epoch+1}/{num_epochs}] Avg Loss: {avg_loss:.4f}\n")

    # overwrite every epoch checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "checkpoints/last.pth")

    # overwrite best epoch checkpoints
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("best model saved")
